In [1]:
import pandas as pd

df = pd.read_csv("Maternal Health Risk Data Set.csv")

df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

### 1.4 stratified train/test split


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Verify stratification actually worked
print("Original:")
print(y.value_counts(normalize=True))

print("Training class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

stratify=y tells Python when you split the dataset, try to keep the same proportion of low-, mid-, and high-risk cases in the training and test sets.

## Model Development

### 2.1 Logistic Regeression

##### 2.1 Logistic Regression Implementation
1. Scale training/test data
2. Apply SMOTE to training data only
3. Create baseline Logistic Regression
4. Hyperparameter tuning
5. Train best Logistic Regression
6. Generate final y_pred_lr

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# 1. Build pipeline:
# Scale -> SMOTE -> Logistic Regression
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("lr", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# 2. Define hyperparameters to test
param_grid_lr = {
    "lr__C": [0.01, 0.1, 1, 10, 100],
    "lr__solver": ["lbfgs", "liblinear"],
    "lr__penalty": ["l2"]
}


# 3. Hyperparameter tuning
grid_search_lr = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid_lr,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)


# 4. Fit on ORIGINAL training data
grid_search_lr.fit(X_train, y_train)


# 5. Get best Logistic Regression pipeline
best_lr_model = grid_search_lr.best_estimator_

print("Best parameters:", grid_search_lr.best_params_)
print("Best CV macro-F1:", grid_search_lr.best_score_)


# 6. Generate predictions for Section 3
y_pred_lr = best_lr_model.predict(X_test)

### 2.2 Random Forest

##### 2.2 Random Forest Implementation
1. Apply SMOTE to training data only
2. Create baseline Random Forest
3. Hyperparameter tuning 
4. Train/use best Random Forest
5. Generate final y_pred_rf

In [ ]:
## NO PIPELINE

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


# 1. Apply SMOTE to the training data only
smote_rf = SMOTE(random_state=42)

X_train_smote_rf, y_train_smote_rf = smote_rf.fit_resample(
    X_train,
    y_train
)


# Check class distribution before and after SMOTE
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote_rf.value_counts())


# 2. Create the Random Forest model
rf_model = RandomForestClassifier(
    random_state=42
)


# 3. Define hyperparameters to test
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}


# 4. Hyperparameter tuning
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_search_rf.fit(
    X_train_smote_rf,
    y_train_smote_rf
)


# 5. Get the best model
best_rf_model = grid_search_rf.best_estimator_

print("Best parameters:", grid_search_rf.best_params_)
print("Best cross-validation macro-F1:", grid_search_rf.best_score_)


# 6. Generate predictions for Section 3 evaluation
y_pred_rf = best_rf_model.predict(X_test)

##### SMOTE before GridSearchCV

In this approach, SMOTE is applied to the entire training set before cross-validation. The balanced training data is then passed into `GridSearchCV` for hyperparameter tuning.

This approach works, but it is less leakage-safe because synthetic samples are created before the cross-validation folds are formed. Related synthetic observations may therefore appear across different folds, which can make validation performance slightly more optimistic.

In [ ]:
## WITH PIPELINE

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Build pipeline
rf_pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

# 2. Hyperparameters to test
param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 20],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}

# 3. Grid search
grid_search_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

# 4. Fit on ORIGINAL training data
grid_search_rf.fit(X_train, y_train)

# 5. Best pipeline
best_rf_model = grid_search_rf.best_estimator_

print("Best parameters:", grid_search_rf.best_params_)
print("Best CV macro-F1:", grid_search_rf.best_score_)

# 6. Final predictions on untouched test data
y_pred_rf = best_rf_model.predict(X_test)

##### SMOTE inside a Pipeline

In this approach, SMOTE and Random Forest are combined inside an `imblearn` Pipeline. During `GridSearchCV`, SMOTE is applied separately within each training fold, while the validation fold remains untouched.

This is more leakage-safe because synthetic samples are generated only from the data available in each training fold. Therefore, this pipeline version is preferred for the final model development and hyperparameter tuning.

**Main difference:** The first approach applies SMOTE before cross-validation, while the pipeline approach applies SMOTE inside each cross-validation training fold. The pipeline approach therefore provides a more reliable estimate of model performance.

## Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import pandas as pd

#### 3.1 Accuracy, Precision, Recall, and Macro-F1

In [ ]:
# Store model predictions
predictions = {
    "Logistic Regression": y_pred_lr,
    "Random Forest": y_pred_rf,
    "SVM": y_pred_svm,
    "Neural Network": y_pred_nn
}

results = []

for model_name, y_pred in predictions.items():

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": macro_f1
    })


results_df = pd.DataFrame(results)

results_df

#### 3.2 Confusion Matrices

In [ ]:
# Confusion matrices
for model_name, y_pred in predictions.items():

    print(f"\n{model_name}")
    print("Confusion Matrix:")

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=["low risk", "mid risk", "high risk"]
    )

    print(cm)


# Classification reports
for model_name, y_pred in predictions.items():

    print(f"\n===== {model_name} =====")

    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )

#### 3.3 High-Risk Recall

In [ ]:
high_risk_results = []

for model_name, y_pred in predictions.items():

    high_risk_recall = recall_score(
        y_test,
        y_pred,
        labels=["high risk"],
        average=None,
        zero_division=0
    )[0]

    high_risk_results.append({
        "Model": model_name,
        "High-Risk Recall": high_risk_recall
    })


high_risk_df = pd.DataFrame(high_risk_results)

high_risk_df

In [ ]:
high_risk_fnr_results = []

for model_name, y_pred in predictions.items():

    actual_high = (y_test == "high risk")
    predicted_high = (y_pred == "high risk")

    # High-risk true positives
    TP = ((actual_high) & (predicted_high)).sum()

    # High-risk false negatives:
    # actually high risk but predicted as another class
    FN = ((actual_high) & (~predicted_high)).sum()

    false_negative_rate = FN / (TP + FN)

    high_risk_fnr_results.append({
        "Model": model_name,
        "High-Risk True Positives": TP,
        "High-Risk False Negatives": FN,
        "High-Risk False Negative Rate": false_negative_rate
    })


high_risk_fnr_df = pd.DataFrame(high_risk_fnr_results)

high_risk_fnr_df

#### 3.4 Final Model Comparison

In [ ]:
final_results_df = results_df.merge(
    high_risk_df,
    on="Model"
)

final_results_df = final_results_df.merge(
    high_risk_fnr_df[
        [
            "Model",
            "High-Risk False Negatives",
            "High-Risk False Negative Rate"
        ]
    ],
    on="Model"
)

final_results_df

#### ^ Model Evaluation

All models are evaluated on the same untouched test set to ensure a fair
comparison. Overall performance is assessed using accuracy, macro precision,
macro recall, and macro F1-score.

Because failure to identify a high-risk pregnancy is particularly important
for this application, high-risk recall and the high-risk false-negative rate
are also evaluated separately. A desirable model should achieve strong
overall classification performance while maximizing high-risk recall and
minimizing high-risk false negatives.